# Decisions and width measures on document images

**Computes:** Two-class FCM and interval memberships on DIBCO 2009-2018: decision identity, F-measure, and the ratio of the Sambuc index of two footprint constructions.

**Inputs:** DIBCO images (downloaded by the notebook).  
**Outputs:** `results/02/results_02.json`, with the per-image values also in `dibco_rows.json`.


In [ ]:
import sys, os, platform

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

RESULTS_DIR = os.environ.get("RESULTS_DIR", "results")
OUTDIR = os.path.join(RESULTS_DIR, "02")
os.makedirs(OUTDIR, exist_ok=True)

OUTDIR = os.path.abspath(OUTDIR)

import numpy as np

print(f"colab       : {IN_COLAB}")
print(f"python      : {platform.python_version()}")
print(f"numpy       : {np.__version__}")
print(f"output dir  : {os.path.abspath(OUTDIR)}")
print()

In [ ]:
import subprocess, glob, os

REPO = "https://github.com/RichSu95/Document_Binarization_Collection.git"
ROOT = os.environ.get("DIBCO_ROOT") or ("/content/dibco_all" if "google.colab" in sys.modules else os.path.abspath(os.path.join(OUTDIR, "dibco_all")))
DATA_DIR = os.path.join(ROOT, "repo", "SauvolaNet", "Dataset")

if not os.path.isdir(DATA_DIR):
    os.makedirs(ROOT, exist_ok=True)
    subprocess.run(["git", "clone", "--depth", "1", "-q", REPO,
                    os.path.join(ROOT, "repo")], check=True)

srcs = glob.glob(os.path.join(DATA_DIR, "*DIBCO*_source.*"))
years = sorted({os.path.basename(s).split("_")[1].replace("DIBCO", "")
                for s in srcs})
print(f"{len(srcs)} DIBCO images across {len(years)} editions: {', '.join(years)}")
assert len(srcs) >= 90, "expected at least 90 DIBCO images"

In [ ]:
import glob, os, json
import numpy as np
from PIL import Image

EPS = 1e-9
M1, M2 = 1.1, 10.1
M_T1 = 2.0
A_REP = 0.15
EPS_BAND = 0.02
ITERS = 30

def _half_pair(u, m):
    r = (1.0 - u) / max(u, EPS)
    return 1.0 / (1.0 + r ** (2.0 / (m - 1.0)))

def load(imgf, gtf):
    I = np.array(Image.open(imgf).convert("L"), dtype=float) / 255.0
    G = np.array(Image.open(gtf).convert("L"))
    return I, (G < 128)

def fcm_u(I, v1, v2, m):
    d1 = np.abs(I - v1) + EPS
    d2 = np.abs(I - v2) + EPS
    return 1.0 / (1.0 + np.clip((d1 / d2) ** (2.0 / (m - 1.0)), 0, 1e12))

def fit(I, m=M_T1, iters=ITERS):
    v1, v2 = np.percentile(I, 5), np.percentile(I, 95)
    for _ in range(iters):
        u = fcm_u(I, v1, v2, m)
        w1, w2 = u ** m, (1 - u) ** m
        v1n = (w1 * I).sum() / max(w1.sum(), EPS)
        v2n = (w2 * I).sum() / max(w2.sum(), EPS)
        if abs(v1n - v1) < 1e-6 and abs(v2n - v2) < 1e-6:
            v1, v2 = v1n, v2n
            break
        v1, v2 = v1n, v2n
    return float(v1), float(v2)

def intervals(I, v1, v2, mode):
    if mode == "pair":
        a, b = fcm_u(I, v1, v2, M1), fcm_u(I, v1, v2, M2)
        return np.minimum(a, b), np.maximum(a, b)
    u = fcm_u(I, v1, v2, M_T1)
    if mode == "power":
        return np.clip(u, 0, 1) ** (1 + A_REP), np.clip(u, 0, 1) ** (1 - A_REP)
    raise ValueError(mode)

def f_measure(pred, gt):
    tp = float((pred & gt).sum())
    p = tp / max(pred.sum(), 1)
    r = tp / max(gt.sum(), 1)
    return 2 * p * r / max(p + r, EPS)

def iou(a, b):
    return float((a & b).sum()) / max(float((a | b).sum()), 1)

PAIRS = sorted(
    (f, f.replace("_source.", "_target."))
    for f in glob.glob(os.path.join(DATA_DIR, "*DIBCO*_source.*"))
)
PAIRS = [(a, b) for a, b in PAIRS if os.path.exists(b)]
print(f"{len(PAIRS)} DIBCO image pairs")

rows = []
print(f"{'image':>22s} {'size':>12s} {'w band':>9s} {'IoU(IT2,T1)':>12s} "
      f"{'F IT2':>7s} {'F T1':>7s} {'rho pair':>8s} {'rho power':>9s} "
      f"{'EB pair':>8s} {'EB power':>8s}")
print("-" * 104)

for f, g in PAIRS:
    key = os.path.basename(f).replace("TRAIN_", "").replace("_source", "")
    key = os.path.splitext(key)[0]
    year = key.split("_")[0].replace("DIBCO", "")
    I, G = load(f, g)
    v1, v2 = fit(I)

    lo_p, hi_p = intervals(I, v1, v2, "pair")
    u2_p = 0.5 * (lo_p + hi_p)
    w_p = hi_p - lo_p
    u_t1 = fcm_u(I, v1, v2, M_T1)

    lo_h = min(_half_pair(0.5, M1), _half_pair(0.5, M2))
    hi_h = max(_half_pair(0.5, M1), _half_pair(0.5, M2))
    w_at_b = float(hi_h - lo_h)
    kk = max(int(0.005 * u2_p.size), 50)
    ord_ = np.argsort(np.abs(u2_p.ravel() - 0.5))[:kk]
    w_band = float(w_p.ravel()[ord_].mean())
    band_frac = float((np.abs(u2_p - 0.5) < EPS_BAND).mean())

    m_it2, m_t1 = u2_p > 0.5, u_t1 > 0.5
    d_id = iou(m_it2, m_t1)

    f_it2, f_t1 = f_measure(m_it2, G), f_measure(m_t1, G)

    def rho_of(w, u2, frac=0.005):
        d = np.abs(u2.ravel() - 0.5)
        k = max(int(frac * d.size), 50)
        o = np.argsort(d)
        near, far = o[:k], o[-k:]
        wr = w.ravel()
        return float(wr[near].mean() / max(wr[far].mean(), EPS))

    lo_w, hi_w = intervals(I, v1, v2, "power")
    w_w = hi_w - lo_w
    u2_w = 0.5 * (lo_w + hi_w)

    r_pair, r_power = rho_of(w_p, u2_p), rho_of(w_w, u2_w)
    eb_pair, eb_power = float(w_p.mean()), float(w_w.mean())

    rows.append(dict(img=key, year=year, shape=list(I.shape), w_boundary=w_at_b,
                     w_band=w_band, band_frac=band_frac,
                     iou_decision=d_id, f_it2=f_it2, f_t1=f_t1,
                     rho_pair=r_pair, rho_power=r_power,
                     eb_pair=eb_pair, eb_power=eb_power,
                     v1=v1, v2=v2, ink_frac=float(G.mean())))
    print(f"{key:>22s} {str(I.shape):>12s} {w_band:9.3e} {d_id:12.6f} "
          f"{f_it2:7.4f} {f_t1:7.4f} {r_pair:8.3f} {r_power:9.3f} "
          f"{eb_pair:8.4f} {eb_power:8.4f}")

print("-" * 104)
n = len(rows)
print(f"images: {n}")
print(f"  footprint at u = 1/2 exactly       : max {max(r['w_boundary'] for r in rows):.2e}")
print(f"  footprint on the nearest 0.5% band : median "
      f"{np.median([r['w_band'] for r in rows]):.4f}  (band is not a point)")
print(f"  pixels within |u-1/2| < 0.02       : median "
      f"{np.median([r['band_frac'] for r in rows])*100:.3f}%")
print(f"  IoU(IT2, type-1) decision          : min {min(r['iou_decision'] for r in rows):.6f}")
print(f"  F-measure, IT2 vs type-1           : "
      f"max |diff| {max(abs(r['f_it2']-r['f_t1']) for r in rows):.2e}")
print(f"  F-measure range                    : "
      f"{min(r['f_t1'] for r in rows):.3f} - {max(r['f_t1'] for r in rows):.3f}")
print(f"  rho, fuzzifier pair                 : median "
      f"{np.median([r['rho_pair'] for r in rows]):.2f}")
print(f"  rho, power                         : median "
      f"{np.median([r['rho_power'] for r in rows]):.0f}")
print()
print(f"  Sambuc index ratio, pair / power  : median "
      f"{np.median([r['eb_pair']/r['eb_power'] for r in rows]):.1f}x")
print()
print("-" * 72)
print(f"{'year':>6s} {'n':>4s} {'min IoU':>9s} {'max |dF|':>10s} "
      f"{'F range':>16s} {'inversion':>10s}")
for y in sorted({r["year"] for r in rows}):
    sub = [r for r in rows if r["year"] == y]
    print(f"{y:>6s} {len(sub):>4d} {min(r['iou_decision'] for r in sub):9.6f} "
          f"{max(abs(r['f_it2']-r['f_t1']) for r in sub):10.2e} "
          f"{min(r['f_t1'] for r in sub):.3f}-{max(r['f_t1'] for r in sub):.3f}      "
          f"{np.median([r['eb_pair']/r['eb_power'] for r in sub]):9.1f}x")
print("-" * 72)
json.dump(rows, open(os.path.join(OUTDIR, "dibco_rows.json"), "w"), indent=1)

In [ ]:
import json as _json, os as _os, time as _time
import numpy as _np
_r = _json.load(open(os.path.join(OUTDIR, "dibco_rows.json")))
_o = dict(notebook="02", hardware="cpu",
          finished=_time.strftime("%Y-%m-%d %H:%M:%S"),
          dataset="DIBCO 2009-2018", n_images=len(_r), rows=_r)
_o["w_at_half_max"] = max(x["w_boundary"] for x in _r)
_o["iou_decision_min"] = min(x["iou_decision"] for x in _r)
_o["f_diff_max"] = max(abs(x["f_it2"] - x["f_t1"]) for x in _r)
_f = _np.array([x["f_t1"] for x in _r])
_o["f_t1_min"], _o["f_t1_max"] = float(_f.min()), float(_f.max())
_o["f_t1_median"] = float(_np.median(_f))
_o["f_t1_q1"], _o["f_t1_q3"] = float(_np.percentile(_f, 25)), float(_np.percentile(_f, 75))
_o["n_f_above_09"] = int((_f > 0.9).sum())
_hi = [x for x in _r if x["f_t1"] > 0.9]
_o["iou_min_among_best"] = min(x["iou_decision"] for x in _hi)
_inv = _np.array([x["eb_pair"] / x["eb_power"] for x in _r])
_o["inversion_min"] = float(_inv.min())
_o["n_inversion_above_1"] = int((_inv > 1).sum())
_o["n_band_empty"] = int(sum(1 for x in _r if x["band_frac"] == 0))
_o["years"] = sorted({x.get("year", "?") for x in _r})
_o["sambuc_inversion_median"] = float(_np.median([x["eb_pair"] / x["eb_power"] for x in _r]))
_o["band_frac_median"] = float(_np.median([x["band_frac"] for x in _r]))

print("=" * 72)
print("02 -- decisions and width measures on document images")
print("=" * 72)
print(f"  images                              : {_o['n_images']} across 8 DIBCO editions")
print(f"  editions covered                    : {', '.join(_o['years'])}")
print(f"  F-measure of the type-1 partition   : median {_o['f_t1_median']:.3f} "
      f"[{_o['f_t1_q1']:.3f}, {_o['f_t1_q3']:.3f}], max {_o['f_t1_max']:.3f}")
print(f"  images with F > 0.9                 : {_o['n_f_above_09']}")
print(f"    of those, minimum IoU             : {_o['iou_min_among_best']:.6f}"
      "")
print()
print(f"  footprint at u = 1/2 exactly        : {_o['w_at_half_max']:.2e}")
print(f"  IoU(IT2 decision, type-1 decision)  : {_o['iou_decision_min']:.6f}  (minimum)")
print(f"  |F(IT2) - F(type-1)|                : {_o['f_diff_max']:.2e}  (maximum)")
print(f"  Sambuc inversion, median            : {_o['sambuc_inversion_median']:.1f}x"
      f"  (minimum {_o['inversion_min']:.1f}x, above 1 on "
      f"{_o['n_inversion_above_1']}/{_o['n_images']})")
print(f"  pixels within |u-1/2| < 0.02        : median "
      f"{_o['band_frac_median']*100:.3f}%, empty on {_o['n_band_empty']}/"
      f"{_o['n_images']} images")
print()
_p = _os.path.join(globals().get("OUTDIR", "."), "results_02.json")
with open(_p, "w") as _f: _json.dump(_o, _f, indent=2)
print()
print("written:", _os.path.abspath(_p))